In [8]:
from notebookutils import mssparkutils

mssparkutils.fs.ls("abfss://<contianer>@<storage_account>.dfs.core.windows.net/")

StatementMeta(divanshpool, 13, 6, Finished, Available, Finished, False)

[FileInfo(path=abfss://bronze@<storage_account>.dfs.core.windows.net/checkpoints, name=checkpoints, size=0),
 FileInfo(path=abfss://<contianer>@<storage_account>.dfs.core.windows.net/ecommerce, name=ecommerce, size=0),
 FileInfo(path=abfss://<contianer>@<storage_account>.dfs.core.windows.net/flight_schedule, name=flight_schedule, size=0),
 FileInfo(path=abfss://<contianer>@<storage_account>.dfs.core.windows.net/ingestion_control, name=ingestion_control, size=0),
 FileInfo(path=abfss://<contianer>@<storage_account>.dfs.core.windows.net/kafka_data, name=kafka_data, size=0),
 FileInfo(path=abfss://<contianer>@<storage_account>.dfs.core.windows.net/source=coinmarketcap, name=source=coinmarketcap, size=0)]

In [9]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("abfss://<contianer>@<storage_account>.dfs.core.windows.net/bronze/sales/year=2026/month=04/")

df.show(5)

StatementMeta(divanshpool, 13, 7, Finished, Available, Finished, False)

+--------+-----------+----------+-----------+-----+--------+------------+--------------+---------+----------+
|order_id|customer_id|   product|   category|price|quantity|total_amount|payment_method|     city|order_date|
+--------+-----------+----------+-----------+-----+--------+------------+--------------+---------+----------+
|      O1|      C4432|Headphones|Electronics|18523|       4|       74092|   Credit Card|Bangalore|2025-12-10|
|      O2|     C61819|Headphones|Accessories|39061|       4|      156244|          Cash|  Chennai|2024-08-08|
|      O3|     C42575|    Laptop|Accessories|18273|       4|       73092|           UPI|   Mumbai|2024-05-11|
|      O4|      C1580|    Laptop|Electronics|40933|       1|       40933|   Net Banking|Hyderabad|2025-03-20|
|      O5|     C23163|    Mobile|Accessories|22798|       2|       45596|   Credit Card|Hyderabad|2025-02-15|
+--------+-----------+----------+-----------+-----+--------+------------+--------------+---------+----------+
only showi

```
Bronze → Silver
```

In [10]:
from pyspark.sql.functions import col

clean_df = df.dropna() \
    .withColumn("price", col("price").cast("int")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("total_amount", col("total_amount").cast("int"))

clean_df.show(5)

StatementMeta(divanshpool, 13, 8, Finished, Available, Finished, False)

+--------+-----------+----------+-----------+-----+--------+------------+--------------+---------+----------+
|order_id|customer_id|   product|   category|price|quantity|total_amount|payment_method|     city|order_date|
+--------+-----------+----------+-----------+-----+--------+------------+--------------+---------+----------+
|      O1|      C4432|Headphones|Electronics|18523|       4|       74092|   Credit Card|Bangalore|2025-12-10|
|      O2|     C61819|Headphones|Accessories|39061|       4|      156244|          Cash|  Chennai|2024-08-08|
|      O3|     C42575|    Laptop|Accessories|18273|       4|       73092|           UPI|   Mumbai|2024-05-11|
|      O4|      C1580|    Laptop|Electronics|40933|       1|       40933|   Net Banking|Hyderabad|2025-03-20|
|      O5|     C23163|    Mobile|Accessories|22798|       2|       45596|   Credit Card|Hyderabad|2025-02-15|
+--------+-----------+----------+-----------+-----+--------+------------+--------------+---------+----------+
only showi

```
Write to Silver
```

In [11]:
clean_df.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://<contianer>@<storage_account>.dfs.core.windows.net/silver/sales_clean/")

StatementMeta(divanshpool, 13, 9, Finished, Available, Finished, False)

```
Silver → Gold
```

In [12]:
from pyspark.sql.functions import sum

gold_df = clean_df.groupBy("product", "city") \
    .agg(sum("total_amount").alias("total_sales"))

gold_df.show()

StatementMeta(divanshpool, 13, 10, Finished, Available, Finished, False)

+----------+---------+-----------+
|   product|     city|total_sales|
+----------+---------+-----------+
|    Camera|Bangalore| 5086739546|
|Headphones|    Delhi| 5014292094|
|        TV|   Mumbai| 5008126934|
|        TV|  Chennai| 5082271980|
|        TV|Bangalore| 5026602884|
|    Tablet|Bangalore| 5025122184|
|    Laptop|   Mumbai| 5005946424|
|    Camera|   Mumbai| 5035072530|
|    Camera|    Delhi| 5034793806|
|        TV|Hyderabad| 5079675656|
|    Camera|  Chennai| 5031747670|
|    Laptop|Bangalore| 5023419950|
|        TV|    Delhi| 5049282426|
|    Mobile|  Chennai| 5028863958|
|Headphones|  Chennai| 5044320956|
|Headphones|Hyderabad| 5030416212|
|    Tablet|    Delhi| 5120677576|
|    Mobile|Bangalore| 5004489054|
|    Tablet|  Chennai| 5031926758|
|    Mobile|    Delhi| 5035843214|
+----------+---------+-----------+
only showing top 20 rows



```
Write to Gold
```

In [13]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://<contianer>@<storage_account>.dfs.core.windows.net/gold/sales_summary/")

StatementMeta(divanshpool, 13, 11, Finished, Available, Finished, False)

```
Validate Output
```

In [14]:
spark.read.format("delta") \
    .load("abfss://<contianer>@<storage_account>.dfs.core.windows.net/gold/sales_summary/") \
    .show()

StatementMeta(divanshpool, 13, 12, Finished, Available, Finished, False)

+----------+---------+-----------+
|   product|     city|total_sales|
+----------+---------+-----------+
|    Camera|Bangalore| 5086739546|
|Headphones|    Delhi| 5014292094|
|        TV|   Mumbai| 5008126934|
|        TV|  Chennai| 5082271980|
|        TV|Bangalore| 5026602884|
|    Tablet|Bangalore| 5025122184|
|    Laptop|   Mumbai| 5005946424|
|    Camera|   Mumbai| 5035072530|
|    Camera|    Delhi| 5034793806|
|        TV|Hyderabad| 5079675656|
|    Camera|  Chennai| 5031747670|
|    Laptop|Bangalore| 5023419950|
|        TV|    Delhi| 5049282426|
|    Mobile|  Chennai| 5028863958|
|Headphones|  Chennai| 5044320956|
|Headphones|Hyderabad| 5030416212|
|    Tablet|    Delhi| 5120677576|
|    Mobile|Bangalore| 5004489054|
|    Tablet|  Chennai| 5031926758|
|    Mobile|    Delhi| 5035843214|
+----------+---------+-----------+
only showing top 20 rows

